# 🍽️ Zomato Restaurant Advanced Analysis
**55,569 restaurants across India** | Ratings, cuisines, cost, geo-analysis

---

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')
print("Libraries loaded")

## 1. Load & Clean Dataset

In [ ]:
df = pd.read_csv("zomato Data Analysis/ZOMATO_FINAL.csv")
df.columns = [c.strip() for c in df.columns]
df["aggregate_rating"]     = pd.to_numeric(df["aggregate_rating"],     errors="coerce")
df["average_cost_for_two"] = pd.to_numeric(df["average_cost_for_two"], errors="coerce")
df["votes"]                = pd.to_numeric(df["votes"],                errors="coerce")
df["latitude"]             = pd.to_numeric(df["latitude"],             errors="coerce")
df["longitude"]            = pd.to_numeric(df["longitude"],            errors="coerce")
print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Summary Statistics

In [ ]:
print("=== ZOMATO DATASET SUMMARY ===")
print(f"Total restaurants  : {len(df):,}")
print(f"Cities             : {df['city'].nunique()}")
print(f"Avg rating         : {df['aggregate_rating'].mean():.2f}")
print(f"Avg cost for two   : ₹{df['average_cost_for_two'].mean():,.0f}")
print(f"Total votes        : {df['votes'].sum():,.0f}")
print("\nRating text distribution:")
print(df["rating_text"].value_counts())
print("\nMissing values:")
print(df.isnull().sum())

## 3. Top Cities — Restaurant Count

In [ ]:
city_stats = df.groupby("city").agg(
    Restaurants=("name","count"),
    Avg_Rating=("aggregate_rating","mean"),
    Avg_Cost=("average_cost_for_two","mean"),
    Total_Votes=("votes","sum"),
).reset_index().sort_values("Restaurants", ascending=False)
city_stats["Avg_Rating"] = city_stats["Avg_Rating"].round(2)

fig = px.bar(city_stats.head(20), x="Restaurants", y="city", orientation="h",
             color="Avg_Rating", color_continuous_scale="RdYlGn",
             color_continuous_midpoint=3.5,
             title="Top 20 Cities — Restaurant Count (colored by Avg Rating)")
fig.update_layout(height=580, yaxis=dict(autorange="reversed"))
fig.show()

## 4. Restaurant Map

In [ ]:
map_df = df.dropna(subset=["latitude","longitude"]).sample(min(8000, len(df)))
fig = px.scatter_mapbox(
    map_df, lat="latitude", lon="longitude",
    color="aggregate_rating", size="votes",
    hover_name="name", hover_data=["city","cuisines","average_cost_for_two"],
    color_continuous_scale="RdYlGn", zoom=4, height=550,
    title="Restaurant Locations (sample 8K)",
    mapbox_style="carto-positron",
)
fig.update_layout(margin=dict(l=0,r=0,t=40,b=0))
fig.show()

## 5. Rating Distribution

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Rating Distribution", "Rating Text Breakdown"])
fig.add_trace(go.Histogram(x=df["aggregate_rating"], nbinsx=25,
    marker_color="steelblue", name="Rating"), row=1, col=1)
rt = df["rating_text"].value_counts()
fig.add_trace(go.Bar(x=rt.index, y=rt.values,
    marker_color="salmon", name="Rating Text"), row=1, col=2)
fig.update_layout(height=420, title="Restaurant Ratings", showlegend=False)
fig.show()

## 6. Cost vs Rating

In [ ]:
fig = px.scatter(
    df.dropna(subset=["aggregate_rating","average_cost_for_two"]).sample(min(8000,len(df))),
    x="average_cost_for_two", y="aggregate_rating",
    color="price_range", hover_name="name",
    trendline="ols", opacity=0.4,
    labels={"average_cost_for_two":"Avg Cost for 2 (₹)","aggregate_rating":"Rating"},
    title="Cost for Two vs Rating (colored by Price Range)")
fig.update_layout(height=480)
fig.show()

## 7. Rating by Price Category

In [ ]:
pr_map = {1:"Budget",2:"Moderate",3:"Expensive",4:"Luxury"}
df["Price Category"] = df["price_range"].map(pr_map)
fig = px.box(df.dropna(subset=["Price Category"]),
             x="Price Category", y="aggregate_rating", color="Price Category",
             title="Rating Distribution by Price Category",
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=420, showlegend=False)
fig.show()

## 8. Top Cuisines

In [ ]:
cuis = df.assign(cuisine=df["cuisines"].str.split(",")).explode("cuisine")
cuis["cuisine"] = cuis["cuisine"].str.strip()

top20_cuis = cuis["cuisine"].value_counts().head(20).reset_index()
top20_cuis.columns = ["Cuisine","Count"]
fig = px.bar(top20_cuis, x="Count", y="Cuisine", orientation="h",
             color="Count", color_continuous_scale="Oranges",
             title="Top 20 Most Common Cuisines")
fig.update_layout(height=550, yaxis=dict(autorange="reversed"))
fig.show()

## 9. Top Rated Cuisines

In [ ]:
cuis_rating = cuis.groupby("cuisine").agg(
    Restaurants=("name","count"),
    Avg_Rating=("aggregate_rating","mean"),
    Avg_Votes=("votes","mean"),
).reset_index()
cuis_rating["Avg_Rating"] = cuis_rating["Avg_Rating"].round(2)

top_rated = cuis_rating[cuis_rating["Restaurants"]>=100].nlargest(20,"Avg_Rating")
fig = px.bar(top_rated, x="Avg_Rating", y="cuisine", orientation="h",
             color="Avg_Rating", color_continuous_scale="Greens",
             title="Top 20 Rated Cuisines (min 100 restaurants)")
fig.update_layout(height=550, yaxis=dict(autorange="reversed"))
fig.show()

## 10. Delivery & Takeaway Analysis

In [ ]:
fig = make_subplots(rows=1, cols=2, specs=[[{"type":"pie"},{"type":"pie"}]])
dlv = df["delivery"].value_counts()
tak = df["takeaway"].value_counts()
fig.add_trace(go.Pie(labels=dlv.index, values=dlv.values, hole=0.4, name="Delivery"), row=1, col=1)
fig.add_trace(go.Pie(labels=tak.index, values=tak.values, hole=0.4, name="Takeaway"), row=1, col=2)
fig.update_layout(title="Delivery vs Takeaway Availability", height=400)
fig.show()

dlv_pct = (df["delivery"]=="Yes").mean()*100
tak_pct = (df["takeaway"]=="Yes").mean()*100
print(f"Delivery available : {dlv_pct:.1f}%")
print(f"Takeaway available : {tak_pct:.1f}%")

## 11. Correlation Matrix

In [ ]:
num_cols = [c for c in ["aggregate_rating","average_cost_for_two","votes",
                        "price_range","No. of Cuisines"] if c in df.columns]
corr = df[num_cols].corr()
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                text_auto=".2f", title="Correlation Matrix", aspect="auto")
fig.update_layout(height=450)
fig.show()

## 12. Most Voted Restaurants

In [ ]:
top_rest = df.nlargest(15,"votes")[
    ["name","city","cuisines","aggregate_rating","average_cost_for_two","votes"]
].reset_index(drop=True)
top_rest.columns = ["Name","City","Cuisines","Rating","Avg Cost (₹)","Votes"]
top_rest

## 13. Key Insights

In [ ]:
print("=== ZOMATO KEY INSIGHTS ===")
print(f"Total restaurants  : {len(df):,}")
print(f"Cities covered     : {df['city'].nunique()}")
print(f"Avg rating         : {df['aggregate_rating'].mean():.2f}")
print(f"Avg cost for 2     : ₹{df['average_cost_for_two'].mean():,.0f}")
top_city = df["city"].value_counts().index[0]
print(f"\nMost restaurants   : {top_city} ({df['city'].value_counts().iloc[0]:,})")
top_cuisine = cuis["cuisine"].value_counts().index[0]
print(f"Most popular cuisine: {top_cuisine}")
pct_good = (df["aggregate_rating"]>=4.0).mean()*100
print(f"Rated >= 4.0       : {pct_good:.1f}%")
print(f"Delivery available : {(df['delivery']=='Yes').mean()*100:.1f}%")